# Smart Farm Autonomous Robot — เทรนและแปลงโมเดลบน Google Colab

โน้ตบุ๊กนี้ทำงานแทนสคริปต์ `tools/train.py`, `tools/evaluate.py` และ `tools/export_tflite.py`
บนเครื่องคอมพิวเตอร์ โดยใช้ GPU ฟรีของ Colab

**ทำไมต้องใช้ Colab:**
1. มี GPU ให้ใช้ฟรี — เทรนเร็วกว่า CPU ราว 20-50 เท่า
2. ขั้นตอนแปลงโมเดลเป็น TFLite (`onnx2tf`, `tensorflow`) ติดตั้งยากบน Windows
   และยังมีปัญหากับ Python 3.13 — บน Colab ใช้ได้ทันทีไม่ต้องตั้งค่าอะไร

**สิ่งที่จะได้เมื่อรันครบทุกเซลล์:**

| ไฟล์ | เอาไปวางที่ | ใช้ทำอะไร |
|---|---|---|
| `best.pt` | `CODE/models/` บน PC | ทดสอบบนเครื่องพัฒนา |
| `best_float32.tflite` | `CODE/models/` บน Raspberry Pi | ใช้งานจริง |
| `best_int8.tflite` | `CODE/models/` บน Raspberry Pi | ทางเลือกที่เร็วกว่า |
| `results.png`, `confusion_matrix.png` | รายงานบทที่ 4 | กราฟผลการเทรน |
| `eval_*.csv` | รายงานบทที่ 4 | ตารางประสิทธิภาพ |

---

## ⚠️ ก่อนเริ่ม — เปิดใช้งาน GPU ก่อน

เมนู **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

ถ้าลืมขั้นตอนนี้ การเทรนจะช้ามาก (หลายชั่วโมงแทนที่จะเป็นสิบนาที)

## ขั้นที่ 0 — ตรวจสภาพแวดล้อมและติดตั้งไลบรารี

In [ ]:
import subprocess
import sys

print("Python :", sys.version.split()[0])

# ตรวจว่าได้ GPU มาหรือยัง
try:
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
    print("GPU    :", gpu if gpu else "ไม่พบ")
except FileNotFoundError:
    gpu = ""
    print("GPU    : ไม่พบ")

if not gpu:
    print("\n" + "!" * 70)
    print("ยังไม่ได้เปิด GPU — ไปที่เมนู Runtime > Change runtime type > T4 GPU")
    print("รันต่อได้ แต่การเทรนจะใช้เวลานานมาก")
    print("!" * 70)

In [ ]:
%pip install -q ultralytics

import torch
import ultralytics

print("ultralytics :", ultralytics.__version__)
print("torch       :", torch.__version__)
print("ใช้ GPU ได้  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("ชื่อ GPU     :", torch.cuda.get_device_name(0))

## ขั้นที่ 1 — ตั้งค่าโปรเจกต์

ค่าเหล่านี้ต้องตรงกับ `config.yaml` ในโปรเจกต์ โดยเฉพาะ **IMGSZ = 320**
ตามที่กำหนดไว้ในเอกสารโครงงาน

In [ ]:
from pathlib import Path

# --- ค่าที่ต้องตรงกับ config.yaml ---
IMGSZ = 320              # ขนาดภาพเข้าโมเดล ห้ามเปลี่ยน ต้องตรงกับที่ใช้บน Raspberry Pi
BASE_MODEL = "yolov8n.pt"  # YOLOv8 Nano
EPOCHS = 120
BATCH = 32               # GPU ของ Colab รับได้สบาย (บน PC ทั่วไปใช้ 16)
PATIENCE = 30            # หยุดก่อนกำหนดถ้าไม่ดีขึ้นภายในกี่รอบ
RUN_NAME = "yolov8n_320"

# ชื่อคลาสวัชพืช ใช้ตอนเลือกค่า confidence ที่เหมาะที่สุด
WEED_CLASS_NAME = "weed"

# --- โฟลเดอร์ทำงาน ---
WORK_DIR = Path("/content/smartfarm")
DATASET_ROOT = WORK_DIR / "datasets"
OUTPUT_DIR = WORK_DIR / "output"
for folder in (WORK_DIR, DATASET_ROOT, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"ขนาดภาพ      : {IMGSZ}x{IMGSZ}")
print(f"โมเดลตั้งต้น  : {BASE_MODEL}")
print(f"จำนวนรอบ     : {EPOCHS}")
print(f"โฟลเดอร์ทำงาน : {WORK_DIR}")

## ขั้นที่ 2 — นำเข้าชุดข้อมูล

เลือกวิธีเดียวจาก 3 วิธี โดยแก้ค่า `METHOD` ในเซลล์ถัดไป

| วิธี | เหมาะกับ | สิ่งที่ต้องเตรียม |
|---|---|---|
| `"roboflow"` | ดาวน์โหลดตรงจาก Roboflow | API Key + ชื่อ workspace/project |
| `"upload"` | มีไฟล์ zip อยู่ในเครื่องแล้ว | ไฟล์ .zip รูปแบบ YOLOv8 |
| `"drive"` | ไฟล์อยู่ใน Google Drive แล้ว | พาธไฟล์ zip ใน Drive |

**หา API Key ของ Roboflow ได้ที่:** roboflow.com → Settings → Roboflow API → Private API Key

**หาชื่อ workspace/project/version ได้จาก URL ของชุดข้อมูล:**
`https://universe.roboflow.com/`**`workspace-name`**`/`**`project-name`**`/dataset/`**`3`**

In [ ]:
import shutil
import subprocess
import sys
import zipfile

METHOD = "roboflow"   # เลือก: "roboflow" | "upload" | "drive"

# ===== ใช้เมื่อ METHOD = "roboflow" =====
ROBOFLOW_API_KEY = ""        # <-- ใส่ API Key ของคุณ
ROBOFLOW_WORKSPACE = ""      # <-- เช่น "my-workspace"
ROBOFLOW_PROJECT = ""        # <-- เช่น "strawberry-weed-detection"
ROBOFLOW_VERSION = 1

# ===== ใช้เมื่อ METHOD = "drive" =====
DRIVE_ZIP_PATH = "/content/drive/MyDrive/smartfarm/dataset.zip"


def extract_zip(zip_path: Path, target: Path) -> None:
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)
    print(f"แตกไฟล์ {zip_path.name} ไปที่ {target}")


dataset_dir = None

if METHOD == "roboflow":
    if not ROBOFLOW_API_KEY:
        raise SystemExit("กรุณาใส่ ROBOFLOW_API_KEY ก่อน หรือเปลี่ยน METHOD เป็น 'upload' / 'drive'")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
    from roboflow import Roboflow

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    downloaded = project.version(ROBOFLOW_VERSION).download(
        "yolov8", location=str(DATASET_ROOT / "roboflow")
    )
    dataset_dir = Path(downloaded.location)

elif METHOD == "upload":
    from google.colab import files

    print("เลือกไฟล์ .zip ของชุดข้อมูล (รูปแบบ YOLOv8)")
    uploaded = files.upload()
    for filename in uploaded:
        if filename.lower().endswith(".zip"):
            target = DATASET_ROOT / "uploaded"
            if target.exists():
                shutil.rmtree(target)
            target.mkdir(parents=True)
            extract_zip(Path(filename), target)
            dataset_dir = target
            break
    else:
        raise SystemExit("ไม่พบไฟล์ .zip ในสิ่งที่อัปโหลดมา")

elif METHOD == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.is_file():
        raise SystemExit(f"ไม่พบไฟล์ {zip_path} ใน Google Drive")
    target = DATASET_ROOT / "from_drive"
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    extract_zip(zip_path, target)
    dataset_dir = target

else:
    raise SystemExit(f"METHOD ไม่ถูกต้อง: {METHOD}")

print(f"\nโฟลเดอร์ชุดข้อมูล: {dataset_dir}")

## ขั้นที่ 3 — ตรวจสอบและซ่อมชุดข้อมูล

เซลล์นี้ทำงานเดียวกับ `tools/dataset_check.py` บนเครื่อง แต่เพิ่มการ **ซ่อมพาธใน `data.yaml`**
ให้ด้วย

ไฟล์ `data.yaml` ที่ Roboflow สร้างมามักใช้พาธแบบ `../train/images` ซึ่งชี้ผิดที่เมื่อ
มาอยู่บน Colab ทำให้ ultralytics หาภาพไม่เจอและฟ้องว่า "ไม่มีภาพสำหรับเทรน"
เซลล์นี้จะเขียนพาธใหม่เป็นแบบเต็มให้อัตโนมัติ

In [ ]:
import yaml
from collections import Counter

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# --- หาไฟล์ data.yaml (บางชุดข้อมูลซ่อนอยู่ในโฟลเดอร์ย่อยอีกชั้น) ---
candidates = sorted(dataset_dir.rglob("data.yaml"))
if not candidates:
    raise SystemExit(f"ไม่พบไฟล์ data.yaml ใน {dataset_dir}")
data_yaml_path = candidates[0]
root = data_yaml_path.parent
print(f"พบ data.yaml ที่ : {data_yaml_path}\n")

with data_yaml_path.open("r", encoding="utf-8") as fh:
    data = yaml.safe_load(fh)

names = data.get("names")
if isinstance(names, dict):
    CLASS_NAMES = [names[k] for k in sorted(names)]
else:
    CLASS_NAMES = list(names or [])
if not CLASS_NAMES:
    raise SystemExit("data.yaml ไม่มีรายชื่อคลาส (names)")


def find_split_dir(split: str):
    """หาโฟลเดอร์ภาพของแต่ละ split แบบยืดหยุ่น รองรับหลายโครงสร้าง"""
    value = data.get(split)
    tried = []
    if value:
        tried += [root / str(value), Path(str(value))]
    tried += [root / split / "images", root / split]
    for candidate in tried:
        try:
            candidate = candidate.resolve()
        except OSError:
            continue
        if candidate.is_dir() and any(p.suffix.lower() in IMAGE_EXT for p in candidate.rglob("*")):
            return candidate
    return None


splits = {}
for split, aliases in (("train", ["train"]), ("val", ["val", "valid"]), ("test", ["test"])):
    found = None
    for alias in aliases:
        found = find_split_dir(alias)
        if found:
            break
    if found:
        splits[split] = found

if "train" not in splits or "val" not in splits:
    raise SystemExit(f"ต้องมีทั้ง train และ val แต่พบเพียง {list(splits)}")

# --- เขียน data.yaml ใหม่ด้วยพาธแบบเต็ม ---
fixed = {"path": str(root), "names": CLASS_NAMES, "nc": len(CLASS_NAMES)}
for split, folder in splits.items():
    fixed[split] = str(folder)

with data_yaml_path.open("w", encoding="utf-8") as fh:
    yaml.safe_dump(fixed, fh, allow_unicode=True, sort_keys=False)

DATA_YAML = str(data_yaml_path)
print("ซ่อมพาธใน data.yaml เรียบร้อย:")
print(yaml.safe_dump(fixed, allow_unicode=True, sort_keys=False))

In [ ]:
# --- นับจำนวนภาพและกล่องของแต่ละคลาส ---
def labels_dir_for(images_dir: Path) -> Path:
    parts = list(images_dir.parts)
    if "images" in parts:
        index = len(parts) - 1 - parts[::-1].index("images")
        parts[index] = "labels"
        return Path(*parts)
    return images_dir.parent / "labels"


problems = []
total_boxes = Counter()
header = f"{'split':<8}{'images':>8}{'boxes':>8}" + "".join(f"{n:>14}" for n in CLASS_NAMES)
print(header)
print("-" * len(header))

for split, images_dir in splits.items():
    labels_dir = labels_dir_for(images_dir)
    image_files = sorted(p for p in images_dir.rglob("*") if p.suffix.lower() in IMAGE_EXT)

    counts = Counter()
    missing = 0
    empty = 0
    for image_path in image_files:
        label_path = (labels_dir / image_path.relative_to(images_dir)).with_suffix(".txt")
        if not label_path.is_file():
            missing += 1
            continue
        lines = [ln for ln in label_path.read_text(encoding="utf-8").split("\n") if ln.strip()]
        if not lines:
            empty += 1
            continue
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                try:
                    counts[int(float(parts[0]))] += 1
                except ValueError:
                    problems.append(f"[{split}] label ผิดรูปแบบ: {label_path.name} -> {line}")

    total_boxes.update(counts)
    row = f"{split:<8}{len(image_files):>8}{sum(counts.values()):>8}"
    row += "".join(f"{counts.get(i, 0):>14}" for i in range(len(CLASS_NAMES)))
    print(row)

    if missing:
        problems.append(f"[{split}] มี {missing} ภาพที่ไม่มีไฟล์ label")
    if empty:
        print(f"         ({split} มีภาพพื้นหลังไม่มีวัตถุ {empty} ภาพ — ปกติดี ช่วยลดการตรวจจับผิดพลาด)")

# --- ตรวจความสมดุลของคลาส ---
if total_boxes:
    most, least = max(total_boxes.values()), min(total_boxes.values())
    if len(total_boxes) < len(CLASS_NAMES):
        missing_classes = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) if i not in total_boxes]
        problems.append(f"คลาสเหล่านี้ไม่มีตัวอย่างเลย: {missing_classes}")
    elif most / max(least, 1) > 5:
        problems.append(
            f"จำนวนตัวอย่างไม่สมดุล ต่างกัน {most/least:.1f} เท่า "
            "— ควรเก็บภาพคลาสที่น้อยเพิ่ม ไม่งั้นโมเดลจะเอนเอียง"
        )

print()
if problems:
    print("พบข้อควรแก้ไข:")
    for i, p in enumerate(problems, 1):
        print(f"  {i}. {p}")
else:
    print("ชุดข้อมูลผ่านการตรวจสอบ พร้อมเทรน")

# --- บอกค่าที่ต้องไปแก้ใน config.yaml ---
print("\n" + "=" * 70)
print("ไปแก้ใน CODE/config.yaml ให้ตรงกับชุดข้อมูลนี้:")
print("=" * 70)
print("classes:")
print(f"  names: {CLASS_NAMES}")
print("=" * 70)

if WEED_CLASS_NAME not in CLASS_NAMES:
    print(f"\n[เตือน] ไม่พบคลาส '{WEED_CLASS_NAME}' ในชุดข้อมูล")
    print(f"        แก้ค่า WEED_CLASS_NAME ให้เป็นหนึ่งใน {CLASS_NAMES}")

## ขั้นที่ 4 — เทรนโมเดล

**สำคัญ: เทรนที่ 320x320 ตั้งแต่แรก** ไม่ใช่เทรน 640 แล้วค่อยลดตอนใช้งาน
เพราะโมเดลเรียนรู้รายละเอียดในระดับความละเอียดที่ใช้ตอนเทรน
ถ้าเทรน 640 แล้วรัน 320 บน Raspberry Pi ความแม่นยำจะตกอย่างเห็นได้ชัด

ค่าการเพิ่มความหลากหลายของข้อมูล (augmentation) ตั้งให้ตรงกับ `tools/train.py`:
- `hsv_v=0.5` สุ่มปรับความสว่างมากกว่าค่าปกติ เพราะแสงในโรงเรือนเปลี่ยนตลอดวัน
- `degrees=10` กล้องติดบนรถอาจเอียงเล็กน้อยเวลาวิ่งผ่านร่องขรุขระ
- `flipud=0.0` ไม่พลิกบน-ล่าง เพราะต้นพืชขึ้นจากพื้นเสมอ

เวลาที่ใช้โดยประมาณบน GPU T4: **ราว 15-40 นาที** ขึ้นกับจำนวนภาพ

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=str(WORK_DIR / "runs"),
    name=RUN_NAME,
    exist_ok=True,
    # --- การเพิ่มความหลากหลายของข้อมูล (ตรงกับ tools/train.py) ---
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    close_mosaic=10,
)

SAVE_DIR = Path(results.save_dir)
BEST_PT = SAVE_DIR / "weights" / "best.pt"
print(f"\nเทรนเสร็จแล้ว")
print(f"โมเดลที่ดีที่สุด : {BEST_PT}")

In [ ]:
# --- แสดงกราฟผลการเทรน (บันทึกภาพเหล่านี้ไปใส่รายงานบทที่ 4 ได้เลย) ---
from IPython.display import Image, display

for filename, caption in (
    ("results.png", "กราฟการเรียนรู้ (loss และ metric แต่ละรอบ)"),
    ("confusion_matrix.png", "Confusion Matrix — ดูว่าโมเดลสับสนระหว่างคลาสไหน"),
    ("val_batch0_pred.png", "ตัวอย่างผลการทำนายบนชุดตรวจสอบ"),
):
    path = SAVE_DIR / filename
    if path.is_file():
        print(f"\n{caption}")
        display(Image(filename=str(path), width=900))
    else:
        print(f"(ไม่พบไฟล์ {filename})")

## ขั้นที่ 5 — ประเมินประสิทธิภาพ

ทำสองอย่าง เหมือน `tools/evaluate.py`:

1. **วัดค่ามาตรฐาน** — Precision, Recall, mAP50, mAP50-95 ทั้งภาพรวมและรายคลาส
2. **ไล่ทดสอบค่า confidence** เพื่อหาค่าที่เหมาะที่สุดสำหรับคลาสวัชพืช

ข้อ 2 สำคัญกับงานนี้เป็นพิเศษ เพราะการตรวจผิดสองแบบมีราคาไม่เท่ากัน:

| ความผิดพลาด | ผลที่ตามมา |
|---|---|
| พ่นทั้งที่ไม่ใช่วัชพืช (False Positive) | เปลืองยา และสารเคมีอาจโดนพืชหลักจนเสียหาย |
| ไม่พ่นทั้งที่เป็นวัชพืช (False Negative) | วัชพืชเหลือรอด ค่อยพ่นรอบหน้าได้ |

งานนี้จึงให้น้ำหนักกับการลด False Positive มากกว่า ควรตั้งค่า confidence ค่อนข้างสูง

In [ ]:
import csv
import json

best_model = YOLO(str(BEST_PT))
class_names_from_model = [best_model.names[k] for k in sorted(best_model.names)]


def extract_metrics(res):
    """ดึงตัวเลขออกจากผลลัพธ์ของ ultralytics"""
    box = res.box
    p, r = float(box.mp), float(box.mr)
    overall = {
        "precision": round(p, 4),
        "recall": round(r, 4),
        "f1": round(2 * p * r / (p + r), 4) if (p + r) else 0.0,
        "mAP50": round(float(box.map50), 4),
        "mAP50-95": round(float(box.map), 4),
    }
    per_class = []
    for position, class_index in enumerate(list(box.ap_class_index)):
        class_index = int(class_index)
        name = class_names_from_model[class_index] if class_index < len(class_names_from_model) else str(class_index)

        def at(attr):
            try:
                return float(getattr(box, attr)[position])
            except Exception:
                return 0.0

        cp, cr = at("p"), at("r")
        per_class.append({
            "class": name,
            "precision": round(cp, 4),
            "recall": round(cr, 4),
            "f1": round(2 * cp * cr / (cp + cr), 4) if (cp + cr) else 0.0,
            "mAP50": round(at("ap50"), 4),
            "mAP50-95": round(at("ap"), 4),
        })
    return overall, per_class


metrics = best_model.val(data=DATA_YAML, imgsz=IMGSZ, split="val", conf=0.001, verbose=False)
overall, per_class = extract_metrics(metrics)

print("=" * 70)
print(" ภาพรวมทุกคลาส")
print("=" * 70)
for key, label in (
    ("precision", "Precision (ที่ทายว่าใช่ ถูกจริงกี่ %)"),
    ("recall", "Recall (ของจริงทั้งหมด จับได้กี่ %)"),
    ("f1", "F1-score"),
    ("mAP50", "mAP@0.5"),
    ("mAP50-95", "mAP@0.5:0.95"),
):
    print(f"  {label:<42} {overall[key]:.4f}")

map50 = overall["mAP50"]
level = ("ดีมาก — ใช้งานจริงได้" if map50 >= 0.90 else
         "ดี — ใช้งานได้ในสภาพแวดล้อมที่ควบคุมได้" if map50 >= 0.80 else
         "พอใช้ — ควรเก็บภาพเพิ่ม" if map50 >= 0.70 else
         "ต่ำ — ตรวจสอบชุดข้อมูลและ label")
print(f"\n  ระดับคุณภาพ: {level}")

print("\n" + "=" * 70)
print(" แยกรายคลาส")
print("=" * 70)
cols = ["class", "precision", "recall", "f1", "mAP50", "mAP50-95"]
print("".join(f"{c:>14}" for c in cols))
for row in per_class:
    print("".join(f"{str(row[c]):>14}" for c in cols))

# บันทึกไว้ใส่รายงาน
with (OUTPUT_DIR / "eval_per_class.csv").open("w", newline="", encoding="utf-8-sig") as fh:
    writer = csv.DictWriter(fh, fieldnames=cols)
    writer.writeheader()
    writer.writerows(per_class)

In [ ]:
# --- ไล่ทดสอบค่า confidence เพื่อหาจุดที่เหมาะที่สุด (ใช้เวลาสักครู่) ---
sweep_rows = []
conf_value = 0.25
while conf_value <= 0.75 + 1e-9:
    res = best_model.val(data=DATA_YAML, imgsz=IMGSZ, split="val",
                         conf=round(conf_value, 2), verbose=False)
    o, pc = extract_metrics(res)
    row = {"conf": round(conf_value, 2), "precision": o["precision"],
           "recall": o["recall"], "f1": o["f1"]}
    for entry in pc:
        if entry["class"] == WEED_CLASS_NAME:
            row["weed_precision"] = entry["precision"]
            row["weed_recall"] = entry["recall"]
            row["weed_f1"] = entry["f1"]
            break
    sweep_rows.append(row)
    print(f"  conf {row['conf']:.2f} -> P {row['precision']:.3f}  R {row['recall']:.3f}  F1 {row['f1']:.3f}")
    conf_value += 0.05

columns = ["conf", "precision", "recall", "f1"]
if sweep_rows and "weed_f1" in sweep_rows[0]:
    columns += ["weed_precision", "weed_recall", "weed_f1"]

print("\n" + "".join(f"{c:>16}" for c in columns))
for row in sweep_rows:
    print("".join(f"{str(row.get(c, '')):>16}" for c in columns))

with (OUTPUT_DIR / "eval_conf_sweep.csv").open("w", newline="", encoding="utf-8-sig") as fh:
    writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(sweep_rows)

score_key = "weed_f1" if (sweep_rows and "weed_f1" in sweep_rows[0]) else "f1"
best_row = max(sweep_rows, key=lambda r: r.get(score_key, 0.0))

print("\n" + "=" * 70)
print(" ไปแก้ใน CODE/config.yaml")
print("=" * 70)
print("classes:")
print(f"  names: {CLASS_NAMES}")
if score_key == "weed_f1":
    print("  per_class_conf:")
    print(f"    {WEED_CLASS_NAME}: {best_row['conf']:.2f}")
else:
    print(f"\nmodel:\n  conf: {best_row['conf']:.2f}")
print("=" * 70)

with (OUTPUT_DIR / "eval_report.json").open("w", encoding="utf-8") as fh:
    json.dump({"imgsz": IMGSZ, "classes": CLASS_NAMES, "overall": overall,
               "per_class": per_class, "conf_sweep": sweep_rows},
              fh, ensure_ascii=False, indent=2)
print(f"\nบันทึกผลไว้ที่ {OUTPUT_DIR}")

## ขั้นที่ 6 — แปลงเป็น TensorFlow Lite

นี่คือเหตุผลหลักที่ต้องมาทำบน Colab สายการแปลง (`onnx` → `onnx2tf` → `tflite`)
ต้องใช้ไลบรารีหลายตัวที่ติดตั้งยากบน Windows แต่บน Colab ใช้ได้เลย

| รูปแบบ | ขนาด | ความเร็วบน Pi 4 | ความแม่นยำ |
|---|---|---|---|
| `float32` | ~12 MB | 7-10 FPS | อ้างอิง 100% |
| `int8` | ~3 MB | 12-18 FPS | ~95-98% |

**แนะนำให้แปลงทั้งสองแบบ** แล้วไปวัดจริงบน Pi ด้วย `tools/benchmark.py`

> **ถ้าเซลล์นี้ error:** มักเกิดจากไลบรารีที่ ultralytics เพิ่งติดตั้งเพิ่มให้
> ให้ไปที่เมนู **Runtime → Restart session** แล้วรันเซลล์นี้ใหม่ได้เลย
> **ไม่ต้องเทรนซ้ำ** — เซลล์นี้จะหาไฟล์ที่เทรนไว้แล้วกลับมาเองอัตโนมัติ

In [ ]:
import shutil
from pathlib import Path

from ultralytics import YOLO

EXPORT_INT8 = True   # ตั้งเป็น False ถ้าต้องการแค่ float32

# --- กู้ตัวแปรคืน กรณีเพิ่งกด Restart session มา (ไม่ต้องเทรนใหม่) ---
try:
    BEST_PT
except NameError:
    print("ไม่พบตัวแปรจากเซลล์ก่อนหน้า -> กำลังค้นหาไฟล์ที่เทรนไว้แล้ว...\n")
    import yaml

    IMGSZ = 320
    WEED_CLASS_NAME = "weed"
    WORK_DIR = Path("/content/smartfarm")
    OUTPUT_DIR = WORK_DIR / "output"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    weights_found = sorted(WORK_DIR.rglob("weights/best.pt"))
    if not weights_found:
        raise SystemExit("ไม่พบไฟล์ best.pt — ต้องรันเซลล์เทรนก่อน")
    BEST_PT = weights_found[-1]
    SAVE_DIR = BEST_PT.parent.parent

    yaml_found = sorted((WORK_DIR / "datasets").rglob("data.yaml"))
    if not yaml_found:
        raise SystemExit("ไม่พบไฟล์ data.yaml — ต้องรันเซลล์นำเข้าชุดข้อมูลก่อน")
    DATA_YAML = str(yaml_found[0])
    with open(DATA_YAML, "r", encoding="utf-8") as fh:
        loaded_names = yaml.safe_load(fh).get("names", [])
    CLASS_NAMES = (
        [loaded_names[k] for k in sorted(loaded_names)]
        if isinstance(loaded_names, dict)
        else list(loaded_names)
    )
    print(f"  โมเดล     : {BEST_PT}")
    print(f"  data.yaml : {DATA_YAML}")
    print(f"  คลาส      : {CLASS_NAMES}\n")

exported_files = {}


def locate_tflite(source_pt: Path, keyword: str):
    """หาไฟล์ .tflite ที่ ultralytics สร้างไว้ (ชื่อไฟล์ต่างกันตามเวอร์ชัน)"""
    search_dirs = [source_pt.parent / f"{source_pt.stem}_saved_model", source_pt.parent]
    fallback = None
    for directory in search_dirs:
        if not directory.is_dir():
            continue
        for candidate in sorted(directory.glob("*.tflite")):
            if keyword in candidate.name:
                return candidate
            fallback = fallback or candidate
    return fallback


# --- float32 ---
print("กำลังแปลงเป็น float32 (ครั้งแรกจะช้าเพราะต้องติดตั้งเครื่องมือแปลง)...\n")
YOLO(str(BEST_PT)).export(format="tflite", imgsz=IMGSZ)

found = locate_tflite(BEST_PT, "float32")
if found:
    destination = OUTPUT_DIR / "best_float32.tflite"
    shutil.copy2(found, destination)
    exported_files["float32"] = destination
    print(f"\nได้ไฟล์ {destination.name}  ({destination.stat().st_size / 1e6:.1f} MB)")
else:
    print("\n[เตือน] หาไฟล์ .tflite แบบ float32 ไม่เจอ")

In [ ]:
# --- int8 (ต้องใช้ภาพจากชุดข้อมูลจริงมาปรับเทียบ) ---
if EXPORT_INT8:
    print("กำลังแปลงเป็น int8...\n")
    try:
        YOLO(str(BEST_PT)).export(format="tflite", imgsz=IMGSZ, int8=True, data=DATA_YAML)
        found = locate_tflite(BEST_PT, "int8")
        if found:
            destination = OUTPUT_DIR / "best_int8.tflite"
            shutil.copy2(found, destination)
            exported_files["int8"] = destination
            print(f"\nได้ไฟล์ {destination.name}  ({destination.stat().st_size / 1e6:.1f} MB)")
        else:
            print("\n[เตือน] หาไฟล์ .tflite แบบ int8 ไม่เจอ")
    except Exception as exc:
        print(f"\n[เตือน] แปลง int8 ไม่สำเร็จ: {exc}")
        print("ไม่เป็นไร ใช้ float32 ได้ตามปกติ")
else:
    print("ข้ามการแปลง int8 (EXPORT_INT8 = False)")

# คัดลอกไฟล์ .pt และกราฟไปรวมไว้ที่เดียว
shutil.copy2(BEST_PT, OUTPUT_DIR / "best.pt")
for filename in ("results.png", "confusion_matrix.png", "results.csv", "args.yaml"):
    source = SAVE_DIR / filename
    if source.is_file():
        shutil.copy2(source, OUTPUT_DIR / filename)

print("\nไฟล์ทั้งหมดใน", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {path.name:<28} {path.stat().st_size / 1e6:8.2f} MB")

## ขั้นที่ 7 — ตรวจสอบไฟล์ TFLite ก่อนนำไปใช้

อย่าเพิ่งเชื่อว่าแปลงสำเร็จเพราะไม่มี error เซลล์นี้จะโหลดไฟล์ `.tflite` ขึ้นมาจริงๆ
แล้วตรวจว่า:

1. รูปร่างอินพุตเป็น `(1, 320, 320, 3)` ตรงกับที่ `src/detector.py` คาดไว้
2. รูปร่างเอาต์พุตมีแกนขนาด `4 + จำนวนคลาส` — ถ้าไม่ตรง โค้ดฝั่ง Pi จะฟ้อง error
3. รันทดสอบได้จริงและใช้เวลาเท่าไร

การเจอปัญหาตรงนี้ดีกว่าไปเจอตอนอยู่หน้า Raspberry Pi มาก

In [ ]:
import time
import numpy as np
import tensorflow as tf

for label, path in exported_files.items():
    print("=" * 70)
    print(f" ตรวจสอบ {path.name}")
    print("=" * 70)

    interpreter = tf.lite.Interpreter(model_path=str(path))
    interpreter.allocate_tensors()
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    in_shape = tuple(int(v) for v in input_detail["shape"])
    out_shape = tuple(int(v) for v in output_detail["shape"])
    print(f"  อินพุต  : {in_shape}  dtype={np.dtype(input_detail['dtype']).name}")
    print(f"  เอาต์พุต : {out_shape}  dtype={np.dtype(output_detail['dtype']).name}")

    ok = True
    if in_shape != (1, IMGSZ, IMGSZ, 3):
        print(f"  [ผิดพลาด] อินพุตควรเป็น (1, {IMGSZ}, {IMGSZ}, 3)")
        ok = False

    expected = 4 + len(CLASS_NAMES)
    if expected not in out_shape:
        print(f"  [ผิดพลาด] เอาต์พุตควรมีแกนขนาด {expected} (= 4 + {len(CLASS_NAMES)} คลาส)")
        ok = False

    # ลองรันจริงด้วยภาพสุ่ม
    dummy = np.zeros(in_shape, dtype=input_detail["dtype"])
    interpreter.set_tensor(input_detail["index"], dummy)
    interpreter.invoke()   # อุ่นเครื่อง

    started = time.perf_counter()
    for _ in range(10):
        interpreter.set_tensor(input_detail["index"], dummy)
        interpreter.invoke()
    elapsed_ms = (time.perf_counter() - started) / 10 * 1000

    print(f"  เวลาต่อเฟรม (บน CPU ของ Colab) : {elapsed_ms:.1f} ms")
    print(f"  ผลการตรวจ : {'ผ่าน — พร้อมใช้บน Raspberry Pi' if ok else 'ไม่ผ่าน'}")
    print()

print("หมายเหตุ: CPU ของ Colab เร็วกว่า Raspberry Pi 4 ราว 3-5 เท่า")
print("ตัวเลขจริงบน Pi ให้วัดด้วย  python tools/benchmark.py")

## ขั้นที่ 8 — ดาวน์โหลดผลลัพธ์

เลือกได้ 2 ทาง: ดาวน์โหลดลงเครื่องโดยตรง หรือบันทึกเข้า Google Drive
(แนะนำให้ทำทั้งสองอย่าง — ไฟล์โมเดลคือผลลัพธ์ที่ใช้เวลาเทรนนานที่สุด ควรมีสำรองไว้)

In [ ]:
# --- ทางที่ 1: บีบไฟล์แล้วดาวน์โหลดลงเครื่อง ---
archive_path = shutil.make_archive(str(WORK_DIR / "smartfarm_model"), "zip", str(OUTPUT_DIR))
print(f"สร้างไฟล์ {archive_path}  ({Path(archive_path).stat().st_size / 1e6:.1f} MB)")

from google.colab import files
files.download(archive_path)

In [ ]:
# --- ทางที่ 2: บันทึกเข้า Google Drive ---
SAVE_TO_DRIVE = True
DRIVE_TARGET = "/content/drive/MyDrive/smartfarm/models"

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    target = Path(DRIVE_TARGET)
    target.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            shutil.copy2(path, target / path.name)
            print(f"  บันทึก {path.name}")
    print(f"\nไฟล์ทั้งหมดถูกบันทึกไว้ที่ {target}")
else:
    print("ข้ามการบันทึกเข้า Drive")

## ขั้นที่ 9 — นำไปใช้ต่อ

### บนเครื่อง PC

แตกไฟล์ `smartfarm_model.zip` แล้ววางไฟล์ตามนี้:

```
CODE/
└── models/
    ├── best.pt                 <- ทดสอบบน PC
    ├── best_float32.tflite     <- ส่งต่อไป Raspberry Pi
    └── best_int8.tflite
```

แก้ `CODE/config.yaml` ตามค่าที่โน้ตบุ๊กนี้พิมพ์ให้ (ในขั้นที่ 3 และ 5):

```yaml
classes:
  names: [...]              # คัดลอกจากผลลัพธ์ขั้นที่ 3
  per_class_conf:
    weed: 0.55              # คัดลอกจากผลลัพธ์ขั้นที่ 5
```

ทดสอบบน PC:

```powershell
python -m src.main --weights models/best.pt
python tools\benchmark.py
```

### บน Raspberry Pi

```bash
# รันที่เครื่อง PC เพื่อส่งไฟล์
scp models/best_float32.tflite pi@<ไอพีของ Pi>:~/CODE/models/
```

แก้ `config.yaml` บน Pi:

```yaml
camera:
  source: picamera2
model:
  weights: models/best_float32.tflite
  num_threads: 4
runtime:
  show_window: false
```

แล้วทดสอบทีละส่วนตามลำดับ:

```bash
python tools/camera_test.py --source picamera2   # กล้องเห็นภาพไหม ภาพคมพอไหม
python tools/relay_test.py                       # ถอดสายปั๊มออกก่อน ฟังแค่เสียงคลิก
python tools/benchmark.py                        # ได้กี่ FPS จริง
python -m src.main --headless                    # รันระบบเต็ม
```

---

### ไฟล์สำหรับรายงานบทที่ 4

| ไฟล์ | ใช้ทำอะไร |
|---|---|
| `confusion_matrix.png` | ตารางแสดงว่าโมเดลสับสนระหว่างคลาสไหน |
| `results.png` | กราฟการเรียนรู้ตลอดการเทรน |
| `eval_per_class.csv` | ตาราง Precision / Recall / mAP รายคลาส |
| `eval_conf_sweep.csv` | ตารางเปรียบเทียบค่า confidence |
| `eval_report.json` | ข้อมูลดิบทั้งหมด |

---

### ถ้าผลยังไม่ดีพอ

| อาการ | สาเหตุที่พบบ่อย | วิธีแก้ |
|---|---|---|
| mAP50 ต่ำกว่า 0.7 | ภาพน้อยเกินไป | เก็บภาพเพิ่มเป็น 500+ ภาพต่อคลาส |
| Precision ต่ำ (พ่นผิดบ่อย) | เกณฑ์ confidence ต่ำไป | เพิ่ม `per_class_conf.weed` และ `spray.confirm_frames` |
| Recall ต่ำ (พ่นไม่ครบ) | เกณฑ์สูงไป หรือวัชพืชในภาพเล็กมาก | ลด conf หรือขยับกล้องให้ใกล้พืชขึ้น |
| ผลดีบน Colab แต่แย่ของจริง | ภาพเทรนต่างจากสภาพจริงมาก | ถ่ายภาพจากโรงเรือนจริงเพิ่ม 100-200 ภาพแล้วเทรนใหม่ |